## profiliing

In [2]:
import os
import pandas as pd
import numpy as np

DATA_DIR = "./Data"
train_path = os.path.join(DATA_DIR, "train.csv")
test_path = os.path.join(DATA_DIR, "test.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("train shape", train_df.shape)
print("test shape", test_df.shape)

display(train_df.head)
display(test_df.head)

train shape (891, 12)
test shape (418, 11)


<bound method NDFrame.head of      PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                     

<bound method NDFrame.head of      PassengerId  Pclass                                          Name  \
0            892       3                              Kelly, Mr. James   
1            893       3              Wilkes, Mrs. James (Ellen Needs)   
2            894       2                     Myles, Mr. Thomas Francis   
3            895       3                              Wirz, Mr. Albert   
4            896       3  Hirvonen, Mrs. Alexander (Helga E Lindqvist)   
..           ...     ...                                           ...   
413         1305       3                            Spector, Mr. Woolf   
414         1306       1                  Oliva y Ocana, Dona. Fermina   
415         1307       3                  Saether, Mr. Simon Sivertsen   
416         1308       3                           Ware, Mr. Frederick   
417         1309       3                      Peter, Master. Michael J   

        Sex   Age  SibSp  Parch              Ticket      Fare Cabin Embarked  
0 

## row type, missing rate, only value

In [3]:
def profile_table(df: pd.DataFrame) -> pd.DataFrame:
    total = len(df)
    mis_cnt = df.isnull().sum()
    mis_rate = mis_cnt / total * 100
    nunique = df.nunique(dropna=True)
    dtypes = df.dtypes.astype(str)

    prof = pd.DataFrame(
        {
            "dtpye": dtypes,
            "nunique": nunique,
            "mis_rate": mis_rate,
            "mis_cnt": mis_cnt,

        }
    ).sort_values(by=["mis_rate", "nunique"], ascending=[False, False])

    example = {}
    for col in df.columns:
        s = df[col].dropna()
        example[col] = s.iloc[0] if len(s) > 0 else np.nan
    prof["example_value"] = pd.Series(example)

    return prof

train_prof = profile_table(train_df)
test_prof  = profile_table(test_df)

display(train_prof)
display(test_prof)

,dtpye,nunique,mis_rate,mis_cnt,example_value
Cabin,object,147,77.104377,687,C85
Age,float64,88,19.865320,177,22.0
Embarked,object,3,0.224467,2,S
PassengerId,int64,891,0.000000,0,1
Name,object,891,0.000000,0,"Braund, Mr. Owen Harris"
Ticket,object,681,0.000000,0,A/5 21171
Fare,float64,248,0.000000,0,7.25
SibSp,int64,7,0.000000,0,1
Parch,int64,7,0.000000,0,0
Pclass,int64,3,0.000000,0,3


,dtpye,nunique,mis_rate,mis_cnt,example_value
Cabin,object,76,78.229665,327,B45
Age,float64,79,20.574163,86,34.5
Fare,float64,169,0.239234,1,7.8292
PassengerId,int64,418,0.000000,0,892
Name,object,418,0.000000,0,"Kelly, Mr. James"
Ticket,object,363,0.000000,0,330911
Parch,int64,8,0.000000,0,0
SibSp,int64,7,0.000000,0,0
Pclass,int64,3,0.000000,0,3
Embarked,object,3,0.000000,0,Q


## abnormal value

In [4]:
num_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric cols:", num_cols)

display(train_df[num_cols].describe().T)

for col in ["Age", "Fare"]:
    if col in train_df.columns:
        s = train_df[col]
        print(f"\n[{col}] min={s.min()} max={s.max()} missing={s.isna().sum()}")
        display(train_df[[col]].sort_values(col).head(5))
        display(train_df[[col]].sort_values(col).tail(5))


Numeric cols: ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']


,count,mean,std,min,25%,50%,75%,max
PassengerId,891.0,446.000000,257.353842,1.00,223.5000,446.0000,668.5,891.0000
Survived,891.0,0.383838,0.486592,0.00,0.0000,0.0000,1.0,1.0000
Pclass,891.0,2.308642,0.836071,1.00,2.0000,3.0000,3.0,3.0000
Age,714.0,29.699118,14.526497,0.42,20.1250,28.0000,38.0,80.0000
SibSp,891.0,0.523008,1.102743,0.00,0.0000,0.0000,1.0,8.0000
Parch,891.0,0.381594,0.806057,0.00,0.0000,0.0000,0.0,6.0000
Fare,891.0,32.204208,49.693429,0.00,7.9104,14.4542,31.0,512.3292



[Age] min=0.42 max=80.0 missing=177


,Age
803,0.42
755,0.67
644,0.75
469,0.75
78,0.83


,Age
859,NaN
863,NaN
868,NaN
878,NaN
888,NaN



[Fare] min=0.0 max=512.3292 missing=0


,Fare
271,0.0
597,0.0
302,0.0
633,0.0
277,0.0


,Fare
438,263.0000
341,263.0000
737,512.3292
258,512.3292
679,512.3292


In [5]:
cat_cols = [c for c in train_df.columns if train_df[c].dtype == "object"]
print("Categorical cols:", cat_cols)

for col in cat_cols:
    vc = train_df[col].value_counts(dropna=False).head(20)
    print(f"\n[{col}] top values (including NaN):")
    display(vc)


Categorical cols: ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']

[Name] top values (including NaN):


Name
Braund, Mr. Owen Harris                                1
Boulos, Mr. Hanna                                      1
Frolicher-Stehli, Mr. Maxmillian                       1
Gilinski, Mr. Eliezer                                  1
Murdlin, Mr. Joseph                                    1
Rintamaki, Mr. Matti                                   1
Stephenson, Mrs. Walter Bertram (Martha Eustis)        1
Elsbury, Mr. William James                             1
Bourke, Miss. Mary                                     1
Chapman, Mr. John Henry                                1
Van Impe, Mr. Jean Baptiste                            1
Leitch, Miss. Jessie Wills                             1
Johnson, Mr. Alfred                                    1
Duff Gordon, Sir. Cosmo Edmund ("Mr Morgan")           1
Taussig, Miss. Ruth                                    1
Jacobsohn, Mrs. Sidney Samuel (Amy Frances Christy)    1
Slabenoff, Mr. Petco                                   1
Harrington, Mr. Charles H 


[Sex] top values (including NaN):


Sex
male      577
female    314
Name: count, dtype: int64


[Ticket] top values (including NaN):


Ticket
347082          7
CA. 2343        7
1601            7
3101295         6
CA 2144         6
347088          6
S.O.C. 14879    5
382652          5
LINE            4
PC 17757        4
17421           4
349909          4
113760          4
4133            4
113781          4
W./C. 6608      4
2666            4
19950           4
347077          4
C.A. 31921      3
Name: count, dtype: int64


[Cabin] top values (including NaN):


Cabin
NaN                687
C23 C25 C27          4
G6                   4
B96 B98              4
C22 C26              3
D                    3
F33                  3
E101                 3
F2                   3
B20                  2
E67                  2
C125                 2
E24                  2
B49                  2
B77                  2
D35                  2
C78                  2
C93                  2
C65                  2
B57 B59 B63 B66      2
Name: count, dtype: int64


[Embarked] top values (including NaN):


Embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64

In [7]:
target_col = "Survived"
if target_col in train_df.columns:
    display(train_df[target_col].value_counts(dropna=False))
    display(train_df[target_col].value_counts(normalize=True).rename("ratio").round(4))


Survived
0    549
1    342
Name: count, dtype: int64

Survived
0    0.6162
1    0.3838
Name: ratio, dtype: float64